# AIg ML walkthrough

This notebook loads the public AIg dataset, shows the schema and sample rows, and reruns the benchmark script to a user-chosen output directory.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts' / 'compare_aig_models.py').exists():
    raise RuntimeError('Open this notebook from the repository root in Jupyter or VS Code.')

DATASET_CSV = PROJECT_ROOT / 'experiments' / 'aig_publication_dataset' / 'aig_publication_dataset.csv'
OUTPUT_DIR = PROJECT_ROOT / 'tmp_model_bench'
COMPARE_SCRIPT = PROJECT_ROOT / 'scripts' / 'compare_aig_models.py'

DATASET_CSV

In [ ]:
df = pd.read_csv(DATASET_CSV)
print('shape:', df.shape)
print('columns:', list(df.columns))
df.head(10)

In [ ]:
display(df['split'].value_counts().rename_axis('split').reset_index(name='rows'))
display(df['scenario'].value_counts().rename_axis('scenario').reset_index(name='rows'))

In [ ]:
numeric_cols = [
    'delta_t_minutes', 'prev_confidence', 'e_telecom', 'e_device', 'e_timing', 'e_ordering',
    'session_step_index', 'recent_conf_mean_3', 'recent_conf_slope_3', 'recent_delta_mean_3',
    'recent_telecom_mean_3', 'recent_device_mean_3', 'recent_timing_mean_3', 'recent_ordering_mean_3',
    'steps_since_strong_telecom', 'minutes_since_strong_telecom', 'aig_label', 'manual_allow',
    'manual_threshold', 'manual_c_value'
]
df[numeric_cols].describe().T

## Rerun the benchmark models

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
cmd = [
    sys.executable,
    str(COMPARE_SCRIPT),
    '--dataset-csv', str(DATASET_CSV),
    '--output-dir', str(OUTPUT_DIR),
]
completed = subprocess.run(cmd, cwd=PROJECT_ROOT, check=True, capture_output=True, text=True)
print('return code:', completed.returncode)
if completed.stdout.strip():
    print(completed.stdout)
if completed.stderr.strip():
    print(completed.stderr)

In [ ]:
comparison = pd.read_csv(OUTPUT_DIR / 'model_comparison.csv')
manual = pd.read_csv(OUTPUT_DIR / 'manual_baseline.csv')
importance = pd.read_csv(OUTPUT_DIR / 'feature_importance_top5.csv')
split_summary = pd.read_csv(OUTPUT_DIR / 'split_summary.csv')

display(split_summary)
display(manual)
display(comparison)

In [ ]:
focus = comparison[comparison['Model'].isin(['logistic_base', 'logistic_rich_no_step'])].copy()
focus.insert(0, 'Display', ['Pointwise logistic', 'History-aware logistic'])
manual_row = pd.DataFrame([
    {
        'Display': 'Manual policy',
        'False step-up rate (benign)': float(manual.loc[0, 'False step-up rate (benign)']),
        'Takeover blocking rate': float(manual.loc[0, 'Takeover blocking rate']),
        'Task completion continuity': float(manual.loc[0, 'Task completion continuity']),
    }
])
focus_metrics = focus[['Display', 'False step-up rate (benign)', 'Takeover blocking rate', 'Task completion continuity']].copy()
plot_df = pd.concat([manual_row, focus_metrics], ignore_index=True)
plot_df

In [ ]:
colors = ['#A04000', '#8C6D46', '#117A65']
metrics = ['False step-up rate (benign)', 'Takeover blocking rate', 'Task completion continuity']

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), constrained_layout=True)
for ax, metric in zip(axes, metrics):
    vals = plot_df[metric].astype(float).tolist()
    labels = plot_df['Display'].tolist()
    bars = ax.bar(range(len(labels)), vals, color=colors, width=0.62)
    ax.set_title(metric, fontsize=15, pad=10)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=0, fontsize=11)
    ax.tick_params(axis='y', labelsize=10)
    ax.grid(axis='y', alpha=0.18)
    ax.set_axisbelow(True)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for bar, value in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f'{value:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.show()

In [ ]:
importance.head(20)